# Logged extraction runs — under a budget

**Every cell below the setup calls the OpenAI API and costs money.** Nothing here is executed
on save.

`build_kg.ipynb` builds one graph by hand; this notebook runs a plan of extractions — models ×
designs × repetitions — one after another, logs each one to `infra/json/kg_runs/<contract>/`
the same way, and stops before any run that could take the total past the budget.
`evaluate_kg.ipynb` then scores them.

What it was used for, all on SteelVault (costs from `runs.json`):

| batch | plan | spent |
|---|---|---|
| Fase 1 smoke test | 1 small call per model | ~$0.03 |
| Fase 4 | gpt-6-luna and gpt-6.1-sol × D1 and D2 × 3 runs | $1.43 |
| boilerplate rule | gpt-6.1-sol × D2 × 3 runs | see `runs.json` |

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'server').exists():
    ROOT = ROOT.parent
SERVER = ROOT / 'server'
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

import hashlib, logging, time
from datetime import datetime

from dotenv import load_dotenv
load_dotenv(SERVER / '.env')

from services.graph.knowledge.extraction import ExtractionIncomplete, build_knowledge_graph
from services.llm.cost_estimator import estimate_model_cost_usd
from services.llm.factory import LLMProviderFactory

logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s')
print('OPENAI key set:', bool(os.getenv('OPENAI_API_KEY')))

## 1. Smoke test — one tiny extraction per model

Two paragraphs, a few cents at most: is the request accepted (no `temperature` where the model
refuses it), is the JSON valid, is the cost priced from the usage the API reports?

In [ ]:
SMOKE_MODELS = ['gpt-6-luna', 'gpt-6.1-sol']
STEM = 'root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement'
src = json.loads((ROOT / 'infra/json/paragraphs' / f'{STEM}.json').read_text())
# 5.4 (approval within 48 hours) and 6. Audit, kept with the outline so the clauses resolve.
sample = [p for p in src['paragraphs'] if p['id'].endswith(('-p-27', '-p-30'))]

for model in SMOKE_MODELS:
    provider = LLMProviderFactory.create('openai', model=model, timeout=300)
    provider.usage.reset()
    graph = build_knowledge_graph(sample, provider, tree=src.get('tree'))
    u = provider.usage
    by_hand = estimate_model_cost_usd(model_name=model, input_tokens=u.prompt, output_tokens=u.completion,
                                      cached_input_tokens=u.cached, cache_write_tokens=u.cache_write)
    print(f'\n=== {model}: {u.calls} calls, prompt {u.prompt} (cached {u.cached}), output {u.completion} '
          f'(reasoning {u.reasoning}), ${u.cost_usd:.5f} — recomputed ${by_hand:.5f}')
    for kind, items in (('obligation', graph.obligations), ('right', graph.rights), ('prohibition', graph.prohibitions)):
        for s in items:
            print(f'  {kind:11} burden={s.burdenPartyId} benefit={s.benefitPartyId} | {s.summary}')

## 2. A batch of logged runs

`PLAN` is a list of `(model, design)`; repeat an entry to repeat the run. Before each run the
projected cost is the priciest run of that model so far (or `PROJECTED` before the first), and
the batch stops if it would take `spent` past `BUDGET`. The first run of an expensive model is
also checked against `FIRST_RUN_CEILING`, so a wrong estimate stops the batch after one run,
not after all of them.

In [ ]:
STEM = 'root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement'
PLAN = [('gpt-6.1-sol', 'D2')] * 3
BUDGET = 1.00
NOTE = ''                                   # what this batch is testing, kept in runs.json
PROJECTED = {'gpt-6-luna': 0.02, 'gpt-6.1-sol': 0.42}
FIRST_RUN_CEILING = {'gpt-6.1-sol': 0.50}

In [ ]:
KNOWLEDGE = SERVER / 'services/graph/knowledge'
RUNS_DIR = ROOT / 'infra/json/kg_runs' / STEM
RUNS_DIR.mkdir(parents=True, exist_ok=True)
src = json.loads((ROOT / 'infra/json/paragraphs' / f'{STEM}.json').read_text())

def fingerprint(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()[:12]

# The prompt is spread over two files: the wording and the shapes it asks for.
prompt_sha = fingerprint((KNOWLEDGE / 'prompts.py').read_bytes() + (KNOWLEDGE / 'ontology.py').read_bytes())

def log_run(kg, model, design, usage, seconds, note):
    run_file = f'{datetime.now():%Y%m%d-%H%M%S}_{model}_{design}.json'
    payload = json.dumps(kg.model_dump(), ensure_ascii=False, indent=2)
    (RUNS_DIR / run_file).write_text(payload, encoding='utf-8')
    manifest_path = RUNS_DIR / 'runs.json'
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else []
    manifest.append({
        'file': run_file, 'model': model, 'reasoningEffort': None, 'pipeline': 'skeleton-v2',
        'design': design, 'prompt': prompt_sha, 'kgSha': fingerprint(payload.encode('utf-8')),
        'createdAt': datetime.now().isoformat(timespec='seconds'), 'costUsd': round(usage.cost_usd, 5),
        'seconds': round(seconds),
        'usage': {'calls': usage.calls, 'prompt': usage.prompt, 'cached': usage.cached, 'cacheWrite': usage.cache_write,
                  'completion': usage.completion, 'reasoning': usage.reasoning},
        'note': note,
    })
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return run_file

spent, priciest, runs_of = 0.0, {}, {}
print(f'prompt {prompt_sha} | budget ${BUDGET:.2f}')
for n, (model, design) in enumerate(PLAN, start=1):
    projected = priciest.get(model, PROJECTED.get(model, BUDGET))
    if spent + projected > BUDGET:
        print(f'STOP before run {n}: spent ${spent:.4f} + projected ${projected:.4f} > ${BUDGET:.2f}')
        break
    provider = LLMProviderFactory.create('openai', model=model, timeout=300)
    provider.usage.reset()
    started, note = time.time(), NOTE
    try:
        kg = build_knowledge_graph(src['paragraphs'], provider, tree=src.get('tree'), design=design)
    except ExtractionIncomplete as exc:
        kg, note = exc.graph, f'{NOTE} — INCOMPLETE: {exc}'
    except Exception as exc:
        spent += provider.usage.cost_usd
        print(f'run {n} {model} {design}: FAILED ({exc}); total ${spent:.4f}')
        break
    u = provider.usage
    spent += u.cost_usd
    priciest[model] = max(priciest.get(model, 0.0), u.cost_usd)
    runs_of[model] = runs_of.get(model, 0) + 1
    run_file = log_run(kg, model, design, u, time.time() - started, note)
    statements = len(kg.obligations) + len(kg.rights) + len(kg.prohibitions)
    print(f'run {n:2d} {model:12s} {design}: ${u.cost_usd:.4f} (total ${spent:.4f}) | {time.time() - started:4.0f}s | '
          f'{statements} statements | {run_file}')
    if runs_of[model] == 1 and u.cost_usd > FIRST_RUN_CEILING.get(model, BUDGET):
        print(f'STOP: the first {model} run cost ${u.cost_usd:.4f}, above its ceiling')
        break
print(f'DONE: spent ${spent:.4f}')